# Alternative Cosmo pipeline for access restrictions

This notebook provides a second `0_` pipeline for the access restrictions campaign.

It replaces the previous `osmium -> ogr2ogr` chain with a direct `cosmo convert` workflow:

1. check that `cosmo` is installed
2. download the Germany extract from Geofabrik if needed
3. validate the YAML filter configuration
4. convert directly from `.osm.pbf` to `.parquet`
5. inspect the resulting GeoParquet file

Output file:

`processed_osm_files/processed_access_restrictions_network_germany_{set_date}.parquet`

In [9]:
import shutil
import subprocess
from pathlib import Path

import geopandas as gpd
import requests

# Keep this in sync with the downstream merge notebook.
#set_date = "260129"
set_date = "260410"

folder_download = Path("../utils/osm_geofabrik_pbf")
folder_processed = Path("processed_osm_files")
filter_config = Path("cosmo_access_restrictions_filter.yaml")
filename = f"germany-{set_date}.osm.pbf"
input_pbf = folder_download / filename
output_parquet = folder_processed / f"processed_access_restrictions_network_germany_cosmoFIL_{set_date}.parquet"
node_cache_mode = "sparse"

folder_download.mkdir(parents=True, exist_ok=True)
folder_processed.mkdir(parents=True, exist_ok=True)

print(f"Input PBF: {input_pbf}")
print(f"Filter config: {filter_config.resolve()}")
print(f"Output parquet: {output_parquet}")

Input PBF: ../utils/osm_geofabrik_pbf/germany-260410.osm.pbf
Filter config: /home/simon/mapillary_trafficsigns/use_cases/access_restrictions_campaign/cosmo_access_restrictions_filter.yaml
Output parquet: processed_osm_files/processed_access_restrictions_network_germany_cosmoFIL_260410.parquet


In [10]:
def run_command(command):
    print("Running:", " ".join(command))
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    return result


def download_geofabrik_pbf(filename, base_url="https://download.geofabrik.de/europe/"):
    file_path = folder_download / filename
    file_url = base_url + filename

    if file_path.exists():
        print(f"File already exists: {file_path}, skipping download.")
        return file_path

    print(f"Downloading: {file_url}")
    response = requests.get(file_url, stream=True, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as file_handle:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file_handle.write(chunk)

    print(f"Downloaded: {file_path}")
    return file_path

In [11]:
cosmo_binary = shutil.which("cosmo")

if not cosmo_binary:
    raise FileNotFoundError(
        "`cosmo` was not found in PATH. Install it first, for example via cargo build/release as described in the upstream repository."
    )

run_command([cosmo_binary, "--version"])
run_command([cosmo_binary, "validate", str(filter_config)])

Running: /home/simon/.cargo/bin/cosmo --version
cosmo 0.1.0

Running: /home/simon/.cargo/bin/cosmo validate cosmo_access_restrictions_filter.yaml
✓ Configuration is valid



CompletedProcess(args=['/home/simon/.cargo/bin/cosmo', 'validate', 'cosmo_access_restrictions_filter.yaml'], returncode=0, stdout='✓ Configuration is valid\n', stderr='')

In [12]:
download_geofabrik_pbf(filename)

if output_parquet.exists():
    print(f"Output already exists: {output_parquet}, skipping conversion.")
else:
    run_command(
        [
            cosmo_binary,
            "convert",
            "--input",
            str(input_pbf),
            "--filters",
            str(filter_config),
            "--output",
            str(output_parquet),
            "--node-cache-mode",
            node_cache_mode,
            "--verbose",
        ]
    )

File already exists: ../utils/osm_geofabrik_pbf/germany-260410.osm.pbf, skipping download.
Running: /home/simon/.cargo/bin/cosmo convert --input ../utils/osm_geofabrik_pbf/germany-260410.osm.pbf --filters cosmo_access_restrictions_filter.yaml --output processed_osm_files/processed_access_restrictions_network_germany_cosmoFIL_260410.parquet --node-cache-mode sparse --verbose
2026-04-11T11:57:47.877852Z  INFO cosmo: Config: 1 table(s), 16 total columns (nodes: false, ways: true, relations: false)
2026-04-11T11:57:47.877955Z  INFO cosmo:   - Table 'access_restrictions_network': 16 columns (nodes: false, ways: true, relations: false)
2026-04-11T11:57:47.877967Z  INFO cosmo::app: Sink: geoparquet -> "processed_osm_files/processed_access_restrictions_network_germany_cosmoFIL_260410.parquet" (16 columns)
2026-04-11T11:57:47.912850Z  INFO cosmo::app: Node cache required: true
2026-04-11T11:57:47.912914Z  INFO cosmo::app: Input size: 4.41 GB
2026-04-11T11:57:47.913496Z  INFO cosmo::app: Node ca

In [5]:
access_network = gpd.read_parquet(output_parquet)

print("number of access-network segments:", len(access_network))
access_network.head()

number of access-network segments: 16287670


,geometry,access,bicycle,foot,hgv,highway,maxaxleload,maxheight,maxlength,maxspeed,maxweight,maxwidth,motor_vehicle,motorcar,motorcycle,osm_id,vehicle,properties
0,"LINESTRING (8.81905 53.04725, 8.81891 53.04722)",None,None,None,None,footway,None,None,None,None,None,None,None,None,None,4392628,None,{}
1,"LINESTRING (13.23646 52.44015, 13.23659 52.440...",None,None,use_sidepath,None,secondary,None,None,None,50,None,None,None,None,None,4392634,None,{}
2,"LINESTRING (13.27894 52.47548, 13.27879 52.475...",None,None,use_sidepath,None,secondary,None,None,None,50,None,None,None,None,None,4392636,None,{}
3,"LINESTRING (13.23602 52.46948, 13.2362 52.46944)",None,None,None,None,service,None,None,None,None,None,None,None,None,None,4392637,None,{}
4,"LINESTRING (13.28357 52.49705, 13.28341 52.497...",None,None,None,None,None,None,default,None,None,None,None,None,None,None,4392641,None,{}


In [6]:
access_network.columns

Index(['geometry', 'access', 'bicycle', 'foot', 'hgv', 'highway',
       'maxaxleload', 'maxheight', 'maxlength', 'maxspeed', 'maxweight',
       'maxwidth', 'motor_vehicle', 'motorcar', 'motorcycle', 'osm_id',
       'vehicle', 'properties'],
      dtype='object')